
# Chapter 14
## Learning to Write Like Shakespeare: Long Short-Term Memory

This chapter explores character-level language modeling and Long Short-Term Memory (LSTM) networks.

The model learns to predict the next character based on the characters that came before it.

The chapter discusses truncated backpropagation, vanishing and exploding gradients, recurrent neural network backpropagation, and the design of LSTM cells.


In [1]:

import numpy as np
import random
import math

np.random.seed(0)
random.seed(0)

print("NumPy version:", np.__version__)
print("Libraries imported successfully.")


NumPy version: 2.1.3
Libraries imported successfully.



## Character Language Modeling

Character language modeling is the task of predicting the next character in a sequence.

Unlike word-level language modeling, which predicts entire words, a character-level model operates on individual characters, including letters, spaces, and punctuation.

The model learns patterns in the training text and uses the preceding characters to estimate the probability of the next character.


In [2]:

import urllib.request

url = (
    "https://raw.githubusercontent.com/"
    "karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)

try:
    with urllib.request.urlopen(url, timeout=30) as response:
        raw = response.read().decode("utf-8")

    print("Dataset berhasil dimuat!")
    print("Jumlah karakter:", len(raw))
    print("\nContoh teks:\n")
    print(raw[:500])

except Exception as e:
    print("Gagal mengunduh dataset:", e)


Dataset berhasil dimuat!
Jumlah karakter: 1115394

Contoh teks:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:

if "raw" not in globals():
    raise RuntimeError(
        "Dataset belum tersedia. Jalankan Cell 4 terlebih dahulu."
    )

vocab = sorted(set(raw))
word2index = {char: i for i, char in enumerate(vocab)}
index2word = {i: char for char, i in word2index.items()}

indices = np.array(
    [word2index[char] for char in raw],
    dtype=np.int32
)

print("Jumlah karakter unik:", len(vocab))
print("Contoh vocabulary:", vocab[:30])
print("Contoh indeks:", indices[:30])


Jumlah karakter unik: 65
Contoh vocabulary: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q']
Contoh indeks: [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56 43  1 61 43
  1 54 56 53 41 43]



## Understanding Character Sequences

The text is converted into numerical indices so that the model can process it.

Each character is assigned an index in the vocabulary. During training, a sequence of characters becomes the input, and the following character becomes the target.

For example, given the input "hell", the model learns to predict the next character in the sequence "hello".

The same vocabulary mapping is used throughout training and text generation.


In [4]:

seq_length = 40

def create_sequences(indices, seq_length=40, stride=1):
    X = []
    y = []

    for i in range(0, len(indices) - seq_length, stride):
        X.append(indices[i:i + seq_length])
        y.append(indices[i + seq_length])

    return np.array(X), np.array(y)


X, y = create_sequences(
    indices[:10000],
    seq_length=seq_length,
    stride=3
)

print("Bentuk X:", X.shape)
print("Bentuk y:", y.shape)

print("Contoh input:")
print("".join(index2word[int(i)] for i in X[0]))

print("Target berikutnya:", index2word[int(y[0])])


Bentuk X: (3320, 40)
Bentuk y: (3320,)
Contoh input:
First Citizen:
Before we proceed any fur
Target berikutnya: t



## Truncated Backpropagation

Backpropagation through time calculates gradients across the sequence steps of a recurrent neural network.

For long sequences, computing gradients through every previous time step can be expensive and difficult to manage.

Truncated backpropagation limits the number of time steps through which gradients are propagated. This reduces memory and computation requirements, although it may limit how much long-range information the model learns.



## Vanishing and Exploding Gradients

During recurrent backpropagation, gradients are repeatedly multiplied through the recurrent computation.

When these multiplications make gradients very small, the network may struggle to learn long-term dependencies. This is known as the vanishing gradient problem.

When gradients become excessively large, training can become unstable. This is known as the exploding gradient problem.

LSTM networks use gates and a cell state to help preserve useful information over longer sequences.


In [5]:

gradient = 1.0
factor = 0.5

for step in range(10):
    gradient *= factor

print("Gradient after repeated multiplication:", gradient)

large_gradient = 1.0

for step in range(10):
    large_gradient *= 2.0

print("Gradient after repeated growth:", large_gradient)


Gradient after repeated multiplication: 0.0009765625
Gradient after repeated growth: 1024.0



## A Toy Example of RNN Backpropagation

A recurrent neural network updates its hidden state using the current input and the previous hidden state.

The hidden state influences the prediction, and the prediction error is propagated backward through the network.

Because the recurrent parameters are reused at every time step, the gradient contributions from different steps must be combined when updating the shared parameters.


In [6]:

input_size = 3
hidden_size = 4

W_xh = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)

sequence = np.random.randn(5, input_size)
hidden = np.zeros(hidden_size)

for t in range(len(sequence)):
    hidden = np.tanh(
        sequence[t].dot(W_xh)
        + hidden.dot(W_hh)
        + b_h
    )

    print(f"Time step {t + 1}: {hidden}")


Time step 1: [ 0.49540808 -0.07575439  0.28383934  0.33084181]
Time step 2: [-0.04958983 -0.00141961 -0.03019051 -0.19173741]
Time step 3: [-0.08446975  0.0466182  -0.00720797  0.10252279]
Time step 4: [0.16696006 0.05612924 0.07394851 0.21952364]
Time step 5: [-0.36315459  0.00054401 -0.24073388 -0.43855084]



## Long Short-Term Memory (LSTM) Cells

An LSTM is a type of recurrent neural network designed to help model long-term dependencies.

An LSTM maintains a hidden state and a cell state. Gates control how information is added, retained, and exposed to the hidden representation.

The three principal gates are the forget gate, input gate, and output gate. Together, they regulate the flow of information through the cell.



## Understanding the LSTM Gates

The forget gate determines which information from the previous cell state should be retained.

The input gate controls how much new information should be added to the cell state.

The output gate controls which information from the cell state contributes to the new hidden state.

These gates use learned parameters and nonlinear activation functions to control information flow.


In [7]:

def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))


class SimpleLSTMCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size

        combined_size = input_size + hidden_size

        self.Wf = np.random.randn(
            combined_size, hidden_size
        ) * 0.1

        self.Wi = np.random.randn(
            combined_size, hidden_size
        ) * 0.1

        self.Wo = np.random.randn(
            combined_size, hidden_size
        ) * 0.1

        self.Wg = np.random.randn(
            combined_size, hidden_size
        ) * 0.1

        self.bf = np.zeros(hidden_size)
        self.bi = np.zeros(hidden_size)
        self.bo = np.zeros(hidden_size)
        self.bg = np.zeros(hidden_size)

    def forward(self, x, h_prev, c_prev):
        combined = np.concatenate([x, h_prev])

        f = sigmoid(combined.dot(self.Wf) + self.bf)
        i = sigmoid(combined.dot(self.Wi) + self.bi)
        o = sigmoid(combined.dot(self.Wo) + self.bo)
        g = np.tanh(combined.dot(self.Wg) + self.bg)

        c = f * c_prev + i * g
        h = o * np.tanh(c)

        return h, c


lstm = SimpleLSTMCell(input_size=3, hidden_size=4)

x = np.array([0.1, 0.2, 0.3])
h_prev = np.zeros(4)
c_prev = np.zeros(4)

h, c = lstm.forward(x, h_prev, c_prev)

print("Hidden state:", h)
print("Cell state:", c)


Hidden state: [ 0.01231971  0.00264785 -0.00039846 -0.00218407]
Cell state: [ 0.02463227  0.00515892 -0.00077006 -0.00429294]


In [8]:

hidden = np.zeros(4)
cell_state = np.zeros(4)

sequence = np.random.randn(5, 3)

for t, x in enumerate(sequence):
    hidden, cell_state = lstm.forward(
        x, hidden, cell_state
    )

    print(f"Step {t + 1}")
    print("Hidden:", hidden)
    print("Cell:", cell_state)


Step 1
Hidden: [-0.03456579 -0.03396481 -0.00024566 -0.01369894]
Cell: [-0.0640121  -0.08825357 -0.00045742 -0.02984791]
Step 2
Hidden: [-0.07729324 -0.03732343 -0.01096323  0.01636379]
Cell: [-0.16065536 -0.08593007 -0.02204047  0.03162706]
Step 3
Hidden: [ 0.0526856   0.00080634  0.0097106  -0.0385376 ]
Cell: [ 0.09815562  0.00142164  0.02018847 -0.08922231]
Step 4
Hidden: [-4.08228494e-02 -3.52420630e-02  8.03840741e-05 -2.18537741e-02]
Cell: [-0.07762301 -0.0950175   0.00014681 -0.04521597]
Step 5
Hidden: [-0.06517775 -0.046417   -0.02267859  0.00242525]
Cell: [-0.12979132 -0.11394802 -0.03674093  0.00439865]



## Generating Text

Once a character language model has learned to predict the next character, it can generate text one character at a time.

The model begins with a starting character or sequence. It predicts a distribution over the possible next characters, selects a character, and feeds that character back into the model.

A trained model can generate text that reflects patterns learned from its training data. A model that has not been trained to minimize a language-modeling loss will not yet generate meaningful Shakespearean text.


In [9]:

sample_index = 0

input_text = "".join(
    index2word[int(i)] for i in X[sample_index]
)

target_char = index2word[int(y[sample_index])]

print("Input sequence:")
print(input_text)

print("\nTarget character:")
print(repr(target_char))


Input sequence:
First Citizen:
Before we proceed any fur

Target character:
't'



## Summary

In this chapter, we explored character-level language modeling and the challenges of training recurrent neural networks over long sequences.

We examined truncated backpropagation, vanishing and exploding gradients, and the flow of information through recurrent states.

We also introduced LSTM cells, which use gates and a cell state to help preserve useful information over time.

The main lesson is that LSTM networks provide a recurrent architecture designed to improve the handling of long-term dependencies in sequential data.
